# Pantheon+ Environment-Dependent H0 Test

**Status:** PRELIMINARY / pipeline lock scaffold  
**Claim level:** Reproducibility support only until Pantheon+ SN table and environment labels are added.

Primary hypothesis:

$$H_0^{void} > H_0^{filament}$$

Primary statistic:

$$\Delta H_0 = H_0^{void} - H_0^{filament}$$

## Required inputs

- `data/pantheon/pantheon_plus.csv`
- `data/pantheon/environment_labels.csv`
- `data/pantheon/pantheon_covariance.txt`

The notebook supports Pantheon-style covariance text files where the first line is `N`, followed by `N*N` covariance entries. It refuses to run the primary fit until the required source tables exist.


In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize_scalar
except Exception:
    minimize_scalar = None

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = Path.cwd().parents[1]

DATA_DIR = REPO_ROOT / "data" / "pantheon"
RESULT_DIR = REPO_ROOT / "observations" / "pantheon-environment-h0" / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SN_TABLE = DATA_DIR / "pantheon_plus.csv"
ENV_TABLE = DATA_DIR / "environment_labels.csv"
COVARIANCE_CANDIDATES = [
    DATA_DIR / "pantheon_covariance.txt",
    DATA_DIR / "Pantheon_cov_subset.txt",
    REPO_ROOT / "Pantheon_cov_subset.txt",
]

OMEGA_M = 0.315
C_KM_S = 299792.458
PRIMARY_Z_MIN = 0.01
PRIMARY_Z_MAX = 0.15
SEED = 42


## Helper functions


In [ ]:
def find_existing(paths: Iterable[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def load_pantheon_covariance(path: Path) -> np.ndarray:
    raw = np.loadtxt(path)
    flat = np.asarray(raw, dtype=float).reshape(-1)
    if len(flat) > 1:
        n0 = int(round(flat[0]))
        if n0 > 0 and len(flat) == 1 + n0 * n0:
            return flat[1:].reshape((n0, n0))

    arr = np.asarray(raw, dtype=float)
    if arr.ndim == 2 and arr.shape[0] == arr.shape[1]:
        return arr

    raise ValueError(
        f"Could not parse covariance matrix from {path}; "
        "expected first-line N plus N*N values or square matrix."
    )


def normalize_id_column(df: pd.DataFrame) -> pd.DataFrame:
    candidates = ["SN_ID", "CID", "cid", "name", "Name", "IAU", "id"]
    for col in candidates:
        if col in df.columns:
            return df.rename(columns={col: "SN_ID"}).copy()
    raise ValueError(f"No recognized SN identifier column found. Columns: {list(df.columns)}")


def normalize_sn_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_id_column(df)
    z_candidates = ["zHD", "z", "redshift", "zCMB"]
    mu_candidates = ["MU_SH0ES", "mu", "MU", "distance_modulus"]

    for col in z_candidates:
        if col in df.columns:
            df = df.rename(columns={col: "z"})
            break
    for col in mu_candidates:
        if col in df.columns:
            df = df.rename(columns={col: "mu"})
            break

    missing = [col for col in ["SN_ID", "z", "mu"] if col not in df.columns]
    if missing:
        raise ValueError(f"Pantheon table missing normalized columns: {missing}. Columns: {list(df.columns)}")
    return df


def normalize_environment_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_id_column(df)
    env_candidates = ["environment", "env", "label", "class"]
    for col in env_candidates:
        if col in df.columns:
            df = df.rename(columns={col: "environment"})
            break
    if "environment" not in df.columns:
        raise ValueError(f"Environment table missing environment column. Columns: {list(df.columns)}")
    df["environment"] = df["environment"].astype(str).str.lower().str.strip()
    return df


def luminosity_distance_mpc(z: np.ndarray, h0: float, omega_m: float = OMEGA_M) -> np.ndarray:
    """Flat LCDM luminosity distance using deterministic trapezoid integration."""
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z, dtype=float)
    omega_l = 1.0 - omega_m

    for i, zi in enumerate(z):
        if zi <= 0:
            out[i] = np.nan
            continue
        grid = np.linspace(0.0, zi, 1024)
        ez = np.sqrt(omega_m * (1.0 + grid) ** 3 + omega_l)
        integral = np.trapz(1.0 / ez, grid)
        dc = (C_KM_S / h0) * integral
        out[i] = (1.0 + zi) * dc
    return out


def distance_modulus_model(z: np.ndarray, h0: float) -> np.ndarray:
    dl_mpc = luminosity_distance_mpc(z, h0)
    return 5.0 * np.log10(dl_mpc) + 25.0


@dataclass(frozen=True)
class H0Fit:
    environment: str
    n: int
    h0: float
    sigma_h0: float
    chi2_min: float


def fit_h0_for_subset(z: np.ndarray, mu_obs: np.ndarray, cov: np.ndarray, environment: str) -> H0Fit:
    cov = np.asarray(cov, dtype=float)
    inv_cov = np.linalg.pinv(cov)

    def chi2(h0: float) -> float:
        mu_model = distance_modulus_model(z, h0)
        resid = mu_obs - mu_model
        return float(resid.T @ inv_cov @ resid)

    if minimize_scalar is not None:
        opt = minimize_scalar(chi2, bounds=(50.0, 90.0), method="bounded")
        h0_best = float(opt.x)
        chi2_min = float(opt.fun)
    else:
        grid = np.linspace(50.0, 90.0, 2001)
        vals = np.array([chi2(v) for v in grid])
        idx = int(np.argmin(vals))
        h0_best = float(grid[idx])
        chi2_min = float(vals[idx])

    scan = np.linspace(max(40.0, h0_best - 10.0), min(100.0, h0_best + 10.0), 4001)
    vals = np.array([chi2(v) for v in scan])
    ok = vals <= chi2_min + 1.0
    sigma = 0.5 * (scan[ok].max() - scan[ok].min()) if np.any(ok) else np.nan

    return H0Fit(environment=environment, n=len(z), h0=h0_best, sigma_h0=float(sigma), chi2_min=chi2_min)


## Input availability check

This cell makes the notebook safe to commit before all Pantheon files are present. If required source files are missing, it prints the exact missing paths and stops before making any scientific claim.


In [ ]:
cov_path = find_existing(COVARIANCE_CANDIDATES)
missing = []

if not SN_TABLE.exists():
    missing.append(str(SN_TABLE.relative_to(REPO_ROOT)))
if not ENV_TABLE.exists():
    missing.append(str(ENV_TABLE.relative_to(REPO_ROOT)))
if cov_path is None:
    missing.append("data/pantheon/pantheon_covariance.txt or Pantheon_cov_subset.txt")

if missing:
    print("Pantheon+ environment-H0 notebook scaffold is installed, but required inputs are missing:")
    for item in missing:
        print(f"  - {item}")
    print("\nNo H0 fit will be run until these files are present.")
else:
    print(f"All required inputs found. Covariance path: {cov_path}")


## Load and merge data

Run remaining cells only after the availability check reports that required files exist.


In [ ]:
if not missing:
    sn = normalize_sn_columns(pd.read_csv(SN_TABLE))
    env = normalize_environment_columns(pd.read_csv(ENV_TABLE))
    cov = load_pantheon_covariance(cov_path)

    merged = sn.reset_index(names="cov_index").merge(
        env[["SN_ID", "environment"]],
        on="SN_ID",
        how="inner",
    )

    merged = merged[(merged["z"] > PRIMARY_Z_MIN) & (merged["z"] < PRIMARY_Z_MAX)]
    merged = merged[merged["environment"].isin(["void", "filament"])].copy()
    merged = merged.sort_values("cov_index").reset_index(drop=True)

    print("Merged primary sample:")
    print(merged["environment"].value_counts())
    print(f"Covariance shape: {cov.shape}")
    display(merged.head())


## Primary covariance-aware H0 fit

Uses the matching covariance submatrix for each environment subset:

$$\chi^2 = (\mu_{obs}-\mu_{model})^T C^{-1}(\mu_{obs}-\mu_{model})$$


In [ ]:
if not missing:
    fits = []

    for environment in ["void", "filament"]:
        subset = merged[merged["environment"] == environment]
        idx = subset["cov_index"].to_numpy(dtype=int)
        c_sub = cov[np.ix_(idx, idx)]

        fit = fit_h0_for_subset(
            subset["z"].to_numpy(dtype=float),
            subset["mu"].to_numpy(dtype=float),
            c_sub,
            environment,
        )
        fits.append(fit)

    fit_table = pd.DataFrame([fit.__dict__ for fit in fits])

    void = fit_table[fit_table["environment"] == "void"].iloc[0]
    filament = fit_table[fit_table["environment"] == "filament"].iloc[0]

    delta_h0 = float(void["h0"] - filament["h0"])
    sigma_delta = float(np.sqrt(void["sigma_h0"] ** 2 + filament["sigma_h0"] ** 2))
    significance = delta_h0 / sigma_delta if sigma_delta > 0 else np.nan

    primary = pd.DataFrame([
        {
            "H0_void": void["h0"],
            "sigma_void": void["sigma_h0"],
            "N_void": int(void["n"]),
            "H0_filament": filament["h0"],
            "sigma_filament": filament["sigma_h0"],
            "N_filament": int(filament["n"]),
            "Delta_H0_void_minus_filament": delta_h0,
            "sigma_delta": sigma_delta,
            "significance_sigma": significance,
            "prediction_direction": "PASS" if delta_h0 > 0 else "FAIL_OR_TENSION",
        }
    ])

    fit_table.to_csv(RESULT_DIR / "pantheon_environment_h0_fit_by_environment.csv", index=False)
    primary.to_csv(RESULT_DIR / "pantheon_environment_h0_primary_result.csv", index=False)

    display(fit_table)
    display(primary)


## Permutation sign test scaffold

This shuffles environment labels while preserving the same redshift-cut sample. It is intentionally simple and should be expanded with redshift-matched permutations before any publication claim.


In [ ]:
def compute_delta_for_labeled_frame(frame: pd.DataFrame, cov: np.ndarray) -> float:
    local_fits = []
    for environment in ["void", "filament"]:
        subset = frame[frame["environment"] == environment]
        idx = subset["cov_index"].to_numpy(dtype=int)
        c_sub = cov[np.ix_(idx, idx)]
        fit = fit_h0_for_subset(
            subset["z"].to_numpy(dtype=float),
            subset["mu"].to_numpy(dtype=float),
            c_sub,
            environment,
        )
        local_fits.append(fit)

    h = {fit.environment: fit.h0 for fit in local_fits}
    return float(h["void"] - h["filament"])


if not missing:
    rng = np.random.default_rng(SEED)
    observed_delta = float(primary["Delta_H0_void_minus_filament"].iloc[0])
    labels = merged["environment"].to_numpy().copy()

    perm_rows = []
    n_perm = 100

    for i in range(n_perm):
        shuffled = merged.copy()
        shuffled["environment"] = rng.permutation(labels)
        try:
            delta = compute_delta_for_labeled_frame(shuffled, cov)
        except Exception:
            delta = np.nan
        perm_rows.append({"permutation": i, "Delta_H0": delta})

    perm = pd.DataFrame(perm_rows)
    perm["abs_ge_observed"] = np.abs(perm["Delta_H0"]) >= abs(observed_delta)
    p_perm = float(perm["abs_ge_observed"].mean())

    perm.to_csv(RESULT_DIR / "pantheon_environment_h0_permutation_null.csv", index=False)

    print(f"Observed Delta_H0 = {observed_delta:.4f}")
    print(f"Approx permutation p-value from {n_perm} shuffles = {p_perm:.4f}")
    display(perm.describe())


## Locked interpretation rules

| Result | Interpretation |
|---|---|
| $\Delta H_0 > 0$, <1σ | correct sign, not significant |
| $\Delta H_0 > 0$, 1–2σ | suggestive |
| $\Delta H_0 > 0$, 2–3σ | interesting, needs replication |
| $\Delta H_0 > 0$, >3σ | strong candidate signal |
| $\Delta H_0 < 0$ | tension with SoCT/PNT prediction |

Do not tune void/filament thresholds or redshift cuts after inspecting $\Delta H_0$.
